In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = (
    SparkSession.builder
    .appName("desafio_08_parquet")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 00:41:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Parte A — CSV → Parquet

In [4]:
df_pedidos = spark.read.csv("/workspace/datasets/08_parquet/pedidos.csv", header= True, inferSchema= True)

In [5]:
df_pedidos.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [6]:
df_pedidos.write.mode("overwrite").option("compression", "snappy").parquet("/workspace/datasets/08_parquet/pedidos_parquet_desafio")

In [7]:
df_pedidos_parquet = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_parquet_desafio")

Qual diferença você percebe entre recuperar o schema de um CSV e recuperar o schema de um Parquet?

Resposta: No csv precisamos avisar o spark sobre o schema atraves da função inferSchema, com isso o spark consegue indentificar a tipagem das colunas, ja com o parquet isso já não é necessario, apenas é necessario passar o caminho do arquivo para a leitura do spark e com isso ele ja consegue definir o schema e os tipos das colunas. 

Parte B — Column Pruning

In [8]:
df_column_pruning = (
    df_pedidos_parquet.select("cliente_id", "valor_total")
)

In [9]:
df_column_pruning.explain(True)

== Parsed Logical Plan ==
'Project ['cliente_id, 'valor_total]
+- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Project [cliente_id#42, valor_total#46]
+- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Optimized Logical Plan ==
Project [cliente_id#42, valor_total#46]
+- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [cliente_id#42,valor_total#46] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_parquet_desafio], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,valor_total:double>



Como o ReadSchema demonstra que ocorreu Column Pruning?

Resposta: Ele demonstra informada que so foram utilizada duas colunas, ao inves de todas do dataframe, colunas essas que estão no select()

Parte C — Predicate Pushdown

In [10]:
df_predicate_pushdown = (
    df_pedidos_parquet.select("cliente_id", "valor_total")
    .filter(F.col("valor_total") > 500)
)

In [11]:
df_predicate_pushdown.explain(True)

== Parsed Logical Plan ==
'Filter ('valor_total > 500)
+- Project [cliente_id#42, valor_total#46]
   +- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Filter (valor_total#46 > cast(500 as double))
+- Project [cliente_id#42, valor_total#46]
   +- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Optimized Logical Plan ==
Project [cliente_id#42, valor_total#46]
+- Filter (isnotnull(valor_total#46) AND (valor_total#46 > 500.0))
   +- Relation [pedido_id#41,cliente_id#42,data_pedido#43,status#44,canal#45,valor_total#46] parquet

== Physical Plan ==
*(1) Filter (isnotnull(valor_total#46) AND (valor_total#46 > 500.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [cliente_id#42,valor_total#46] Batched: true, DataFilters: [isnotnull(valor_total#46), (valor_total#46 > 500.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file

Explique com suas palavras a diferença entre:

- ReadSchema
- PushedFilters

Resposta: Readschema descreve as colunas que serão lidas da fonte para aquela consulta, por exemplo se fizermos df = spark.read.csv(...) no readschema havera todas as colunas do dataframe, se fizermos um filtro selecionando coluna A e B, nesse cenario no readschema existira apenas as colunas A e B, isso é chamado de columnm pruning, quando o spark realmente utiliza as colunas necessarias. Ja PusheFilters é uma otimização do spark onde ele aplica os filtros do codigo diretamente na fonte dos dados, essa tecnica é conhecida como predicate pushdown, onde ele mostra os filtros enviados ao leitor da fonte.

Parte D — Dataset particionado

In [12]:
df_pedidos.write.mode("overwrite").partitionBy("status").parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [13]:
df_pedidos_por_status_desafio = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [14]:
df_pedidos_por_status_desafio.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- status: string (nullable = true)



In [15]:
df_pedidos_por_status_desafio.explain(True)

== Parsed Logical Plan ==
Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Optimized Logical Plan ==
Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<pedido_id:int,cliente_id:int,data_pedido:date,canal:string,valor_total:double>



Se status não aparece no ReadSchema dos arquivos Parquet, de onde o Spark recupera seu valor?

Resposta: Ele recupera atraves das partições que foram criadas, tendo em vista que, o dataframe foi salvo em parquet e particionado pela coluna status, o spark consegue recuperar o valor a partir disso.

Parte E — Partition Pruning

In [16]:
df_status_particionado = (
    df_pedidos_por_status_desafio.filter(F.col("status") == "concluido")
)

In [17]:
df_status_particionado.explain(True)

== Parsed Logical Plan ==
'Filter ('status = concluido)
+- Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Filter (status#77 = concluido)
+- Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Optimized Logical Plan ==
Filter (isnotnull(status#77) AND (status#77 = concluido))
+- Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [isnotnull(status#77), (status#77 = concluido)], PushedFilters: [], ReadSchema: struct<pedido_id:int

Agora explique a diferença entre:

- PartitionFilters
- PushedFilters

Resposta: PartitionFilters, basicamente seleciona as partições que serão lidas, no caso foi utilizado isnotnull status ou seja os dados da coluna status não podem ser nulos e status = concluido, com isso podemos concluir que o partitionfilters evita ler os dados das outras partições, mas ainda pode consultar diretórios e metadados. Ja PushedFilters são filtros que o spark empurra para o leitor

O que o Spark consegue evitar ler graças ao Partition Pruning?

Resposta: A partir do partition pruning o spark consegue evitar leituras desncessarias, por exemplo no meu codigo eu quero somente status = concluido, como a coluna de status foi divida por partições o spark me retorna apenas linhas lidas da partição "concluido" ignorando as demais

Parte F — As otimizações juntas

In [18]:
df_select_parquet = (
    df_status_particionado.select("cliente_id", "valor_total")
    .filter((F.col("status") == "concluido") & (F.col("valor_total") > 500))
)

In [19]:
df_select_parquet.explain(True)

== Parsed Logical Plan ==
'Filter (('status = concluido) AND ('valor_total > 500))
+- Project [cliente_id#73, valor_total#76]
   +- Filter (status#77 = concluido)
      +- Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Project [cliente_id#73, valor_total#76]
+- Filter ((status#77 = concluido) AND (valor_total#76 > cast(500 as double)))
   +- Project [cliente_id#73, valor_total#76, status#77]
      +- Filter (status#77 = concluido)
         +- Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Optimized Logical Plan ==
Project [cliente_id#73, valor_total#76]
+- Filter ((isnotnull(status#77) AND isnotnull(valor_total#76)) AND ((status#77 = concluido) AND (valor_total#76 > 500.0)))
   +- Relation [pedido_id#72,cliente_id#73,data_pedido#74,canal#75,valor_total#76,status#77] parquet

== Physical Plan ==
*(1) Project [cliente_id#73,

Indentifique: 

- Column Pruning

Resposta: O column pruning ocorreu no select(cliente_id, valor_total) onde não há necessidade do spark utilizar todas as colunas do dataframe, ignorando-as, ele utiliza apenas aquelas que estão dentro do select.

- Predicate Pushdown

Resposta: Ocorreu no filtro valor_total > 500 essa otimização se chama predicate pushdown onde o spark envia a condição do filtro ao leitor da fonte para que ele possa evitar leituras desnecessarias

- Partition Pruning

Resposta: O partition pruning ocorre quando o spark a partir de um dataframe ja particionado seleciona apenas umas de suas partições, ignorando todas outras, ou seja, se tivermos um dataframe particionado por status e dentre as opções tivermos "concluido, ausente, enviado..." se caso eu faça um filtro selecionando status = "enviado" o spark vai ignorar os outros dois e me trazer apenas os dados referentes a status = enviado

Parte G — Row Groups e Data Skipping

Imagine:

Row Group A
valor_total:
min = 50
max = 300

Row Group B
valor_total:
min = 100
max = 900

Row Group C
valor_total:
min = 10
max = 400

Filtro:

valor_total > 500

Explique quais Row Groups podem ser descartados utilizando as estatísticas min/max e por quê.

Resposta: Row Groups são uma estrutura de organização dos dados para selecionar grupos que atendem aos filtro, entao por exemplo no filtro valor_total > 500, o spark ira descartar o raw group A e C e apenas considerar o B ja que atende previamente a condição > 500 mas nem todas as linhas passam no filtro, visto que o max = 900

Qual a diferença entre Data Skipping e Partition Pruning?

Resposta: Sobre o data skipping, ainda utilizando o filtro > 500 como exemplo o spark fez um data skipping dos Row Groups A e C ignorando-as completamente. Já partition pruning é exclusivamente voltado a partições, se particionarmos um arquivo por mes (1 ao 12) e fizermos um filtro selecinando os meses de 3 a 6 o spark ira ignorar as outras partições, retornando apenas o meses de março a junho

Parte H — Decisão de particionamento

Imagine uma tabela de vendas com:

800 milhões de registros

pedido_id
cliente_id
ano
mes
estado
valor_total

Cardinalidade aproximada:

pedido_id   → centenas de milhões
cliente_id  → milhões
ano         → 5
mes         → 12
estado      → 27

explique quais colunas você escolheria para partitionBy(...) e quais evitaria

Resposta: Escolheria tres colunas, ano, mes, são colunas que apresentam uma boa cardinalidade, não iram gerar muitos arquivos no storage e alem do mais, são dados de extrema importancia a final de contas, são vetores que sempre haverão de ser consultados, "em determinado ano quantas vendas tivemos", "qual o mes de menor faturamento" tendo em vista essas metricas, essas colunas seriam as que eu particionaria. E por fim acabaria evitando particionar pedido_id e cliente_id ambos tem uma cardinalidade gigantesca podendo gerar small files, ja que provalmente cada id trará um dado distinto isso acabaria gerando uma pasta para cada cliente por exemplo, tornando totalmente inviavel tanto para o spark quanto para o usuario final.

Questão final da Aula 08
Imagine dois pipelines:

Pipeline A

CSV
↓
Spark
↓
CSV

e:

Pipeline B

CSV bruto
↓
Spark
↓
Parquet
↓
consultas posteriores

Explique por que o segundo pode ser uma arquitetura mais adequada quando os dados passam a ser reutilizados repetidamente para processamento analítico.

Resposta: O pipeline B acaba sendo mais vantajoso pois ele utiliza parquet para manipulação de dados, e esse tipo de arquivo é o que se da melhor com o spark, primeiramente, ele ja vem com o schema pre definido, diferentemente do csv que temos que definir a tipagem das colunas atraves do inferSchema, com o parquet não há essa necessidade, o parquet é em formato colunar facilitando a leitura e o processamento do spark, diferentemente do formato csv, onde temos textos e divisoes geralmente por ; ou , o parquet tambem se beneficia de tres otimizações, Column Pruning ocorre quando o spark utiliza apenas as colunas necessarias naquele codigo, ele não utiliza todas que estao no dataframe, o segundo é o Predicate Pushdown onde a partir dos filtros o spark empurra a leitura do codigo para o leitor final, ou seja se utilizarmos um filtro valor > 500 ele ignorará os demais valores e trará apenas os que correspondem ao filtro, economizando processamento, e por ultimo Partition Pruning quando particionamos um dataframe em arquivos parquet, e dentro desse mesmo particionamentos fizermos consultas especificas, o spark ignora as demais partições e ira trazer apenas aquele que foi solicitado